# 01 Smoke Test

Colab validation notebook for the first real D1 run.

- Repo remains the source of truth.
- Notebook only bootstraps the environment, fetches default smoke assets, and calls the repository CLI.
- Primary video path is the official SAM2 video predictor.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/ChiCoTheLaAnh/ProjectFinalCS415.git'
REPO_DIR = '/content/ProjectFinalCS415'

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/ProjectFinalCS415
!bash setup_colab.sh --with-models


In [ ]:
import subprocess
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

import groundingdino
import sam2

print('GroundingDINO import OK:', groundingdino.__file__)
print('SAM2 import OK:', sam2.__file__)


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DRIVE_ROOT = Path('/content/drive/MyDrive/cv-final-project')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
INPUT_DIR = DRIVE_ROOT / 'inputs'
OUTPUT_ROOT = DRIVE_ROOT / 'results' / 'smoke_test'
GROUNDING_CKPT = str(CHECKPOINT_DIR / 'groundingdino_swint_ogc.pth')
SAM2_CKPT = str(CHECKPOINT_DIR / 'sam2.1_hiera_small.pt')

DEFAULT_IMAGE_URL = 'https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/.asset/cat_dog.jpeg'
DEFAULT_VIDEO_URL = 'https://raw.githubusercontent.com/IDEA-Research/Grounded-SAM-2/main/assets/hippopotamus.mp4'
DEFAULT_IMAGE_PATH = INPUT_DIR / 'groundingdino_cat_dog.jpeg'
DEFAULT_VIDEO_PATH = INPUT_DIR / 'grounded_sam2_hippopotamus.mp4'

IMAGE_PROMPT = 'dog'
VIDEO_PROMPT = 'hippopotamus'
MAX_FRAMES = 10
DEVICE = 'cuda'

for path in (CHECKPOINT_DIR, INPUT_DIR, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

if not DEFAULT_IMAGE_PATH.exists():
    urlretrieve(DEFAULT_IMAGE_URL, DEFAULT_IMAGE_PATH)
if not DEFAULT_VIDEO_PATH.exists():
    urlretrieve(DEFAULT_VIDEO_URL, DEFAULT_VIDEO_PATH)

INPUT_IMAGE = str(DEFAULT_IMAGE_PATH)
INPUT_VIDEO = str(DEFAULT_VIDEO_PATH)

print('Image prompt:', IMAGE_PROMPT)
print('Video prompt:', VIDEO_PROMPT)
print('Image path:', INPUT_IMAGE)
print('Video path:', INPUT_VIDEO)
print('Grounding checkpoint:', GROUNDING_CKPT)
print('SAM2 checkpoint:', SAM2_CKPT)


In [ ]:
for required_path in (Path(GROUNDING_CKPT), Path(SAM2_CKPT)):
    if not required_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {required_path}')
    print(required_path, required_path.stat().st_size, 'bytes')


In [ ]:
IMAGE_OUTPUT = str(OUTPUT_ROOT / 'image')

!python3 scripts/run_custom_video.py \
  --config configs/base.yaml \
  --input_video "$INPUT_IMAGE" \
  --prompt "$IMAGE_PROMPT" \
  --output_dir "$IMAGE_OUTPUT" \
  --grounding_ckpt "$GROUNDING_CKPT" \
  --sam2_ckpt "$SAM2_CKPT" \
  --device "$DEVICE" \
  --max_frames 1


In [ ]:
import json
from IPython.display import Image, display

image_overlay = Path(IMAGE_OUTPUT) / 'smoke_image_overlay.png'
image_summary_path = Path(IMAGE_OUTPUT) / 'run_summary.json'

display(Image(filename=str(image_overlay)))
with image_summary_path.open('r', encoding='utf-8') as handle:
    image_summary = json.load(handle)
for required_field in ('model_stack', 'runtime_sec', 'artifacts'):
    if required_field not in image_summary:
        raise KeyError(f'Missing required image summary field: {required_field}')
print(json.dumps(image_summary, indent=2))


In [ ]:
VIDEO_OUTPUT = str(OUTPUT_ROOT / 'video')

!python3 scripts/run_custom_video.py \
  --config configs/base.yaml \
  --input_video "$INPUT_VIDEO" \
  --prompt "$VIDEO_PROMPT" \
  --output_dir "$VIDEO_OUTPUT" \
  --grounding_ckpt "$GROUNDING_CKPT" \
  --sam2_ckpt "$SAM2_CKPT" \
  --device "$DEVICE" \
  --max_frames "$MAX_FRAMES"


In [ ]:
import json
from IPython.display import Video, display

video_overlay = Path(VIDEO_OUTPUT) / 'smoke_video_overlay.mp4'
video_summary_path = Path(VIDEO_OUTPUT) / 'run_summary.json'

display(Video(str(video_overlay), embed=True))
with video_summary_path.open('r', encoding='utf-8') as handle:
    video_summary = json.load(handle)
for required_field in ('model_stack', 'runtime_sec', 'artifacts'):
    if required_field not in video_summary:
        raise KeyError(f'Missing required video summary field: {required_field}')
print(json.dumps(video_summary, indent=2))
print('video_mode:', video_summary['artifacts'].get('video_mode'))
if 'fallback_reason' in video_summary['artifacts']:
    print('fallback_reason:', video_summary['artifacts']['fallback_reason'])
